# Context-Gated Channel Attention for Attention U-Net
### Medical Images Processing with Deep Learning (336033) — Final Project

**Base paper:** Oktay et al., *Attention U-Net: Learning Where to Look for the Pancreas* (MIDL 2018)

**Our extension:** the original attention gate produces only a *spatial* coefficient conditioned on a coarse gating signal `g`. We add a **channel-attention branch that is also conditioned on `g`** (unlike self-attention modules such as CBAM/SE-Net, which use only the feature map's own statistics). We compare four variants that all share one U-Net implementation:

| `attention_type` | Model |
|---|---|
| `None` | Vanilla U-Net (baseline) |
| `spatial` | Attention U-Net (original, reproduced) |
| `cbam` | Attention U-Net + naive CBAM bolt-on (control) |
| `hybrid` | Attention U-Net + **context-gated channel attention (ours)** |

**Dataset:** Medical Segmentation Decathlon — Task09_Spleen (41 labeled 3D abdominal CT volumes), sliced to 2D axial slices for a Colab-feasible proof-of-concept.

> Run cells top to bottom. On a Colab GPU runtime (`Runtime > Change runtime type > T4 GPU`) the full 4-variant × 3-seed experiment takes roughly 1–2 hours; reduce `SEEDS` / `MAX_EPOCHS` in the config cell for a faster smoke run.


## 1. Setup — install dependencies and fetch the source code


In [ ]:
# Install dependencies (torch is preinstalled on Colab).
!pip install -q monai nibabel scikit-image

# Clone the project code from GitHub so the src/ package is available in Colab.
# (Safe to re-run: the '|| true' avoids an error if the folder already exists.)
![ -d 'Medical-Images-Analysis-with-Deep-Learning--Final-project-' ] || git clone https://github.com/AvivNiem/Medical-Images-Analysis-with-Deep-Learning--Final-project-.git
%cd Medical-Images-Analysis-with-Deep-Learning--Final-project-

import os
assert os.path.isdir('src'), 'src/ not found — the clone step above did not run correctly.'
print('Working dir:', os.getcwd())


## 1b. Experiment configuration (QUICK vs FULL)
`QUICK` runs a proof-of-concept scale that finishes in one Colab session. `FULL` is the configuration used for the reported results (run on an L40 GPU). Set `QUICK=False` to reproduce the full experiment where hardware/time allow.


In [ ]:
QUICK = True

if QUICK:
    SEEDS, MAX_EPOCHS, IMG = [0, 1], 30, 128
    SWEEP_SIZES = [8, None]            # low-data sweep sizes (None = all patients)
else:
    SEEDS, MAX_EPOCHS, IMG = [0, 1, 2, 3, 4], 60, 256
    SWEEP_SIZES = [4, 8, 16, None]

print(f'Config: QUICK={QUICK}  seeds={SEEDS}  epochs={MAX_EPOCHS}  image_size={IMG}  sweep_sizes={SWEEP_SIZES}')


## 2. Sanity check — forward-pass shapes for all four variants
This is the shape smoke-test that could not be run in the offline dev sandbox; it confirms every variant builds and produces correctly-shaped output before we spend time training.


In [ ]:
import torch
from src.models import build_model, count_parameters

x = torch.randn(2, 1, 128, 128)
for att in [None, 'spatial', 'cbam', 'hybrid']:
    m = build_model(att)
    out = m(x)
    assert out.shape == (2, 1, 128, 128), (att, out.shape)
    maps = m.get_attention_maps()
    print(f'{str(att):8s}  out={tuple(out.shape)}  params={count_parameters(m):,}')
print('\nAll variants build and produce correct output shapes.')


## 3. Data — download Task09_Spleen and build 2D slices
Downloads the official Decathlon archive via MONAI, applies soft-tissue HU windowing + normalization, slices volumes into 2D axial `.npy` pairs, and keeps all spleen-containing slices plus a sample of background slices. Runs once; re-running is cheap (skips existing files).


In [ ]:
from pathlib import Path
from src.data import download_spleen_dataset, build_2d_slice_dataset

IMAGE_SIZE = IMG                       # from the QUICK/FULL config above
ROOT = Path('./decathlon_data')
SLICES = Path(f'./slices_2d_{IMAGE_SIZE}')   # resolution-specific cache

if not SLICES.exists() or not any(SLICES.iterdir()):
    task_dir = download_spleen_dataset(str(ROOT))
    build_2d_slice_dataset(task_dir, SLICES, image_size=IMAGE_SIZE)
else:
    print('Cached slices already exist at', SLICES)


### Inspect a few slices
Quick visual check that windowing + masks look sensible before training.


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from src.data import patient_level_split, SpleenSliceDataset

train_ids, val_ids, test_ids = patient_level_split(SLICES, val_frac=0.15, test_frac=0.2, seed=1234)
print(f'Patients — train={len(train_ids)} val={len(val_ids)} test={len(test_ids)}')

peek = SpleenSliceDataset(SLICES, train_ids, augment=False)
fg = [i for i in range(len(peek)) if peek[i][1].sum() > 0][:4]
fig, axes = plt.subplots(1, len(fg), figsize=(3*len(fg), 3))
for ax, i in zip(axes, fg):
    img, msk = peek[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.contour(msk.squeeze(), levels=[0.5], colors='deepskyblue')
    ax.axis('off')
plt.suptitle('Sample slices with spleen contour'); plt.tight_layout(); plt.show()


## 4. Train all variants across multiple seeds
`run_experiment` trains every (variant × seed) with Adam + Dice loss, early stopping on validation Dice, and a fixed patient-level split shared across all runs. Results are saved to `results/results.json`.

**Tip:** for a fast first pass set `SEEDS=[0]` and `MAX_EPOCHS=15`. For the reported results use ≥3 seeds.


In [ ]:
from src.train import TrainConfig, run_experiment

cfg = TrainConfig(
    slice_dir=str(SLICES),
    attention_types=[None, 'spatial', 'cbam', 'hybrid'],
    seeds=SEEDS,
    max_epochs=MAX_EPOCHS,
    batch_size=8,
    lr=3e-4,
    grad_clip=1.0,
    loss_name='dice_focal',
    image_size=IMG,
    out_dir='./results',
)

# Main 4-way comparison at the full training set.
results = run_experiment(cfg)


## 5. Results table + statistical significance
Aggregate per-variant metrics (mean ± std across seeds) in the paper's Table-1 format, then run a **paired Wilcoxon signed-rank test** on per-slice Dice — the same kind of significance test the original paper reports.


In [ ]:
from src.stats import aggregate_by_variant, format_results_table

summary = aggregate_by_variant(results)
print(format_results_table(summary))


In [ ]:
# Paired significance test on per-slice test Dice: ours (hybrid) vs each baseline.
# Reuses the models trained by run_experiment (loaded from disk) - no retraining.
import torch, numpy as np
from src.data import SpleenSliceDataset
from src.metrics import dice_score
from src.stats import paired_wilcoxon
from src.train import get_device, load_trained_model

device = get_device()
test_ds = SpleenSliceDataset(SLICES, test_ids, augment=False)
fg_idx = [i for i in range(len(test_ds)) if test_ds[i][1].sum() > 0]

def per_slice_dice(model):
    model.eval(); out = []
    with torch.no_grad():
        for i in fg_idx:
            img, msk = test_ds[i]
            logits = model(img.unsqueeze(0).to(device))
            out.append(dice_score(logits, msk.unsqueeze(0).to(device)).item())
    return out

# Use seed 0 (the first trained seed) for the aligned per-slice comparison.
per_slice = {}
for att in [None, 'spatial', 'cbam', 'hybrid']:
    model = load_trained_model(results, att, seed=cfg.seeds[0], cfg=cfg, device=device)
    per_slice[str(att)] = per_slice_dice(model)

for baseline in ['None', 'spatial', 'cbam']:
    t = paired_wilcoxon(per_slice, 'hybrid', baseline)
    print(f"hybrid vs {baseline:8s}: median Dice diff {t['median_diff']:+.3f}, p = {t['p_value']:.4f}")


## 5b. Low-data regime sweep
Trains every variant at several training-set sizes and plots Dice vs. training size. This reproduces the original paper's key analysis — whether attention's benefit grows as data shrinks — and is where our context-gated channel gate is hypothesised to help most.


In [ ]:
from src.train import run_low_data_sweep
from src.stats import aggregate_sweep
from src.visualize import plot_low_data_curve

sweep = run_low_data_sweep(cfg, train_sizes=SWEEP_SIZES)
sweep_agg = aggregate_sweep(sweep)
plot_low_data_curve(sweep_agg, save_path='results/fig_low_data_curve.png'); plt.show()


## 6. Qualitative visualizations
Prediction overlays, spatial attention maps (paper Fig. 3a/4 style), and our context-gated channel weights.


In [ ]:
from src.visualize import (plot_training_curves, plot_prediction_overlay,
                            plot_spatial_attention, plot_channel_attention, show_worst_cases)

plot_training_curves(results, save_path='results/fig_training_curves.png'); plt.show()


In [ ]:
# Load one trained model per variant (seed 0) from disk for the qualitative panels.
models = {}
for att in [None, 'spatial', 'cbam', 'hybrid']:
    models[str(att)] = load_trained_model(results, att, seed=cfg.seeds[0], cfg=cfg, device=device)

# Pick a foreground test slice and compare predictions across variants.
img, msk = test_ds[fg_idx[len(fg_idx)//2]]
plot_prediction_overlay(models, img, msk, device, save_path='results/fig_overlay.png'); plt.show()


In [ ]:
# Spatial attention maps for the original Attention U-Net.
plot_spatial_attention(models['spatial'], img, device, save_path='results/fig_spatial_attn.png'); plt.show()


In [ ]:
# OUR extension: context-gated channel weights (hybrid variant).
plot_channel_attention(models['hybrid'], img, device, save_path='results/fig_channel_attn.png'); plt.show()


In [ ]:
# Failure analysis: worst-case slices for our model.
show_worst_cases(models['hybrid'], test_ds, device, k=4, save_path='results/fig_worst_cases.png'); plt.show()


## 7. Save everything
All metrics are in `results/results.json`; all figures are saved as PNGs in `results/` for inclusion in the report. If you mounted Drive, copy them there so they persist after the runtime ends.


In [ ]:
import os
print('Saved artifacts in results/:')
for f in sorted(os.listdir('results')):
    print('  ', f)
